## Introduction to Digital Image Processing with Python
The most fundamental libraries we will be using throughout the entire course are:
* **OpenCV**
 * For loading, saving and manipulating images and videos.
* **numpy**
 * Powerful tool for working with (multidimensional) matrices (which are mathematical representations of digital images and videos).
* **matplotlib**
 * Library for easy plotting (figures, plots, images, etc.).

In this notebook we will cover:
* Loading and displaying an image, and why OpenCV uses **BGR** instead of RGB.
* Converting between colour spaces: BGR &harr; RGB, grayscale, HSV and CIELAB.
* Basic array **slicing** to crop an image.
* Alternative image-loading libraries (Pillow).
* Why RGB is not a *perceptual* colour space, using an RGB vs CIELAB comparison.

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]

Let's start by loading and plotting an image.

By default `cv2.imread` returns a 3-channel colour image. A flag can change this, e.g. `cv2.imread(path, cv2.IMREAD_GRAYSCALE)` loads the image directly as grayscale, and `cv2.IMREAD_UNCHANGED` keeps an alpha channel if the file has one.

In [ ]:
img = cv2.imread('data/kodim21.png')
plt.imshow(img)
print('Image shape (rows, cols, channels):', img.shape)

**Important**: For historical reasons, OpenCV loads images in **BGR** colour space (not RGB!). We need to manually change the order of colour channels.

In [ ]:
rows, cols, channels = img.shape
img_rgb = np.zeros_like(img)

for r in range(rows):
    for c in range(cols):
        pixel = img[r, c, :]
        img_rgb[r, c, 0] = pixel[2]
        img_rgb[r, c, 1] = pixel[1]
        img_rgb[r, c, 2] = pixel[0]

plt.imshow(img_rgb)

The loop above is easy to read but slow: it runs in pure Python over every single pixel. The exact same channel reversal can be written as one vectorised numpy operation (reversed slicing).

In [ ]:
img_rgb = img[:, :, ::-1]   # reverse the channel axis: BGR -> RGB
plt.imshow(img_rgb)

OpenCV also provides a large variety of dedicated conversion functions.

In [ ]:
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
plt.imshow(img_rgb)

### Grayscale
`cv2.COLOR_BGR2GRAY` collapses the three colour channels into a single intensity channel using a weighted sum (approximately `0.299 R + 0.587 G + 0.114 B`, which reflects how sensitive the human eye is to each primary colour). We pass `cmap='gray'` so that matplotlib does not apply a false-colour map to the single-channel image.

In [ ]:
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
plt.imshow(img_gray, cmap='gray')

Note that the grayscale image has only **two** dimensions: the channel axis is gone, because there is now a single intensity value per pixel.

In [ ]:
print('Grayscale shape (rows, cols):', img_gray.shape)

### Cropping with array slicing
An image is just a numpy array, so we can crop it by slicing. The first index is the **row** range (vertical, top to bottom) and the second is the **column** range (horizontal, left to right). Here we keep the top half of the rows and the right half of the columns.

In [ ]:
plt.imshow(img_rgb[0:rows // 2, cols // 2:, :])

### Image Loading Alternatives

Apart from OpenCV there are other packages that allow you to read images, e.g., [scikit-image](https://scikit-image.org/docs/0.23.x/api/skimage.io.html#skimage.io.imread), [torchvision](https://pytorch.org/vision/0.19/generated/torchvision.io.read_image.html) or [Pillow](https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.open), among many others.

In [ ]:
from PIL import Image
img_pil = Image.open('data/kodim21.png')
img_pil

Pillow does not load images as numpy arrays, and unlike OpenCV it uses **RGB** channel order. To keep processing the image as an array we simply convert it:

In [ ]:
img_pil_array = np.array(img_pil)
print('Type:', type(img_pil_array), '| shape:', img_pil_array.shape)
plt.imshow(img_pil_array)   # already RGB, no channel swap needed

### RGB vs CIELAB
RGB is not a *perceptual* colour space: the mathematical difference between two colours does not necessarily match how different they look to a human eye. CIELAB is designed so that distance in the colour space approximates perceptual difference.

To compare colours numerically we use the **sum of absolute differences (SAD)** across the three channels, i.e. the L1 distance `|dc1| + |dc2| + |dc3|`. Below we build three colour patches: `color_2` is `color_1` made uniformly brighter, and `color_3` darkens the green channel only.

In [ ]:
color_1 = np.zeros((50, 50, 3), dtype=np.uint8)
color_1[..., 0] = 100
color_1[..., 1] = 200
color_1[..., 2] = 100

# |color_1 - color_2| = 150
color_2 = np.zeros((50, 50, 3), dtype=np.uint8)
color_2[..., 0] = 150
color_2[..., 1] = 250
color_2[..., 2] = 150

# |color_1 - color_3| = 120
color_3 = np.zeros((50, 50, 3), dtype=np.uint8)
color_3[..., 0] = 100
color_3[..., 1] = 80
color_3[..., 2] = 100

plt.subplot(131), plt.imshow(color_1), plt.title('Color 1')
plt.subplot(132), plt.imshow(color_2), plt.title('Color 2')
plt.subplot(133), plt.imshow(color_3), plt.title('Color 3')

In [ ]:
color_1_lab = cv2.cvtColor(color_1, cv2.COLOR_RGB2LAB)
color_2_lab = cv2.cvtColor(color_2, cv2.COLOR_RGB2LAB)
color_3_lab = cv2.cvtColor(color_3, cv2.COLOR_RGB2LAB)

print('Color 1 (LAB)', color_1_lab[0, 0, :])
print('Color 2 (LAB)', color_2_lab[0, 0, :])
print('Color 3 (LAB)', color_3_lab[0, 0, :])
print(' ')

print('SAD with respect to color 1')
print('Color 2', np.sum(np.abs(color_1_lab[0, 0, :].astype(np.float32) - color_2_lab[0, 0, :].astype(np.float32))))
print('Color 3', np.sum(np.abs(color_1_lab[0, 0, :].astype(np.float32) - color_3_lab[0, 0, :].astype(np.float32))))

### What just happened

In **RGB**, the SAD from `color_1` is **150** to `color_2` and **120** to `color_3`, so by the RGB numbers `color_2` is the *farther* colour.

Perceptually it is the other way around: `color_2` is just a brighter version of the same green and looks very close to `color_1`, whereas `color_3` has a visibly different hue. The **CIELAB** SAD reflects this: the distance to `color_2` comes out smaller than the distance to `color_3`, which matches what we actually see.

*Aside:* OpenCV packs 8-bit LAB into the 0&ndash;255 range (`L` is scaled, `a` and `b` are offset by 128), so these values are not in canonical CIELAB units. That does not affect this example, since we only compare the distances against each other.

### HSV
HSV separates *colour* from *brightness*, which makes it the natural choice for colour-based thresholding and segmentation. Its three channels are:

* **H (Hue)**: the colour itself (red, yellow, green, ...), expressed as an angle on the colour wheel. In OpenCV's 8-bit representation the range is `0-179` (degrees divided by 2) and it *wraps around*: 0 and 179 are both red.
* **S (Saturation)**: how *pure* or vivid the colour is, `0-255`. Low saturation looks washed-out or grayish; 0 is a pure shade of gray.
* **V (Value)**: how *bright* the pixel is, `0-255`. At 0 the pixel is black regardless of the other two channels.

Because hue stays roughly constant when the lighting changes, we can select "all the blueish pixels" with a simple range on H, which is awkward to express in RGB.

In [ ]:
img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
h, s, v = cv2.split(img_hsv)

plt.subplot(131), plt.imshow(h, cmap='gray'), plt.title('Hue')
plt.subplot(132), plt.imshow(s, cmap='gray'), plt.title('Saturation')
plt.subplot(133), plt.imshow(v, cmap='gray'), plt.title('Value')

As an example, let's isolate the blue sky. `cv2.inRange` returns a binary mask that is white where *every* channel falls inside the given bounds; we then keep only those pixels of the original image.

In [ ]:
lower = np.array([90, 60, 40])     # blueish hue, reasonably saturated and bright
upper = np.array([130, 255, 255])
mask = cv2.inRange(img_hsv, lower, upper)

masked = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

plt.subplot(131), plt.imshow(img_rgb), plt.title('Original')
plt.subplot(132), plt.imshow(mask, cmap='gray'), plt.title('Sky mask')
plt.subplot(133), plt.imshow(masked), plt.title('Masked')